# OVD-Diagnose — Domain 2: Medical (VinDr chest X-ray)

14 finding classes with boxes. Includes detector-intrinsic `L_det`, bootstrap CIs,
reliability diagrams, qualitative examples, and (partly degenerate) synthetic controls.

**Data:** Kaggle → Add Data → `awsaf49/vinbigdata-512-image-dataset`.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow sentence-transformers

In [ ]:
CSV  = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv'
IMGS = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train'
OUT  = 'data/medical/annotations/instances_val.json'
import os
print('csv exists:', os.path.exists(CSV), '| imgs exists:', os.path.isdir(IMGS))

In [ ]:
!python tools/vindr_to_coco.py --csv "{CSV}" --imgs "{IMGS}" --out "{OUT}" --img_ext .png
from pycocotools.coco import COCO
c = COCO(OUT); bb = [a['bbox'] for a in c.loadAnns(c.getAnnIds())[:8]]
mx = max(v for b in bb for v in b)
print('max box coord:', round(mx), '-> OK' if mx <= 600 else '-> UNSCALED')

In [ ]:
LIMIT = 0   # 0 = full (4394); 200 quick
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/medical/annotations/instances_val.json \
  --imgs "{IMGS}" \
  --out  runs/diag/medical \
  --device cuda:0 {limit_arg} \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
print(pd.read_csv('runs/diag/medical/fingerprints.csv').to_string(index=False))

In [ ]:
import json, os
from tools.ovd_diagnose import bootstrap_ci
ann = 'data/medical/annotations/instances_val.json'
for m in ['yoloworld', 'owlv2', 'groundingdino']:
    p = f'runs/diag/medical/{m}/results_global.json'
    if os.path.exists(p):
        ci = bootstrap_ci(ann, json.load(open(p)), metric='ioa_f1', n_boot=1000)
        print(f"{m:>14}: IoA-F1 = {ci['mean']:.4f}  [{ci['lo']:.4f}, {ci['hi']:.4f}]")

In [ ]:
# Reliability diagrams
!python tools/plot_reliability.py \
  --ann data/medical/annotations/instances_val.json \
  --models yoloworld:runs/diag/medical/yoloworld/results_global.json \
           owlv2:runs/diag/medical/owlv2/results_global.json \
           groundingdino:runs/diag/medical/groundingdino/results_global.json \
  --out paper/figures/reliability_medical

## Qualitative examples — localization failure (missed findings)

In [ ]:
!python tools/qualitative.py \
  --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
  --results runs/diag/medical/owlv2/results_global.json \
  --mode missed --n 6 --out paper/figures/qual_medical.png
from IPython.display import Image
Image('paper/figures/qual_medical.png')

## Validation controls (medical)
**Caveat:** medical AP$\approx$0 → temperature/S controls partly degenerate; blur (L) is meaningful.

In [ ]:
!python tools/synthetic_controls.py --control temperature \
  --ann data/medical/annotations/instances_val.json \
  --results runs/diag/medical/owlv2/results_global.json \
  --out runs/controls/medical_owlv2_temperature.csv
!python tools/synthetic_controls.py --control blur \
  --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
  --sam_weights mobile_sam.pt --out runs/controls/medical_blur.csv --limit 200 --device cuda:0

**Cross-domain read:** on medical, both `L` (SAM) and `L_det` (detector-intrinsic) should be ≈1 —
localization fails under *both* proposers, so the reading is not a SAM artifact.